In [1]:
!pip install spacy

In [2]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 77.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import spacy
nlp = spacy.load('en_core_web_sm')
print('spaCy ready!')

spaCy ready!


Import re and datetime

In [4]:
import re
from datetime import datetime

Extract Dates

In [5]:
def extract_dates(text):
    """Extract dates in various formats"""
    patterns = [
        r'\d{1,2}/\d{1,2}/\d{4}',      # MM/DD/YYYY
        r'\d{1,2}-\d{1,2}-\d{4}',      # DD-MM-YYYY
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}',  # Month DD, YYYY
        r'\d{4}-\d{2}-\d{2}'           # ISO format
    ]
    dates = []
    for pattern in patterns:
        matches = re.findall(pattern, text)
        dates.extend(matches)
    return dates

# Test
text = "Invoice date: 03/15/2024. Due: March 30, 2024"
print(extract_dates(text))

['03/15/2024', 'March 30, 2024']


 Extract Currency Amounts

In [6]:
def extract_amounts(text):
    """Extract currency amounts"""
    pattern = r'\$?\d{1,3}(?:,\d{3})*(?:\.\d{2})?'
    amounts = re.findall(pattern, text)
    cleaned = []
    for amount in amounts:
        clean = amount.replace('$', '').replace(',', '')
        cleaned.append(float(clean))
    return cleaned

# Test
text = "Total: $1,250.50. Tax: $125.05. Subtotal: 1125.45"
print(extract_amounts(text))

[1250.5, 125.05, 112.0, 5.45]


 Extract Invoice/Order Numbers

In [7]:
def extract_invoice_number(text):
    """Extract invoice/order numbers"""
    patterns = [
        r'INV-\d{4}-\d{3}',
        r'#\d{5,}',
        r'ORDER-[A-Z0-9]+',
        r'Invoice (?:Number|#):?\s*([A-Z0-9-]+)'
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1) if match.groups() else match.group(0)
    return None

# Test
text = "Invoice Number: INV-2024-001"
print(extract_invoice_number(text))

INV-2024-001


Basic NER

In [8]:
import spacy

# Load model
nlp = spacy.load('en_core_web_sm')

# Sample invoice text
text = """Invoice from Acme Corporation 
123 Main Street, New York, NY 10001 
Contact: John Smith (john@acme.com) 
Date: March 15, 2024 
Amount Due: $1,250.50"""

# Process text
doc = nlp(text)

# Extract entities
print('Found entities:')
for ent in doc.ents:
    print(f'{ent.text:20} {ent.label_:15} {spacy.explain(ent.label_)}')

Found entities:
Acme Corporation     ORG             Companies, agencies, institutions, etc.
123                  CARDINAL        Numerals that do not fall under another type
Main Street          FAC             Buildings, airports, highways, bridges, etc.
New York             GPE             Countries, cities, states
10001                DATE            Absolute or relative dates or periods
John Smith           PERSON          People, including fictional
March 15, 2024       DATE            Absolute or relative dates or periods
1,250.50             MONEY           Monetary values, including unit


Extract Specific Entity Types

In [9]:
def extract_entities(text):
    """Extract and organize entities by type"""
    doc = nlp(text)
    entities = {
        'persons': [],
        'organizations': [],
        'locations': [],
        'dates': [],
        'money': []
    }
    for ent in doc.ents:
        if ent.label_ == 'PERSON':
            entities['persons'].append(ent.text)
        elif ent.label_ == 'ORG':
            entities['organizations'].append(ent.text)
        elif ent.label_ in ['GPE', 'LOC']:
            entities['locations'].append(ent.text)
        elif ent.label_ == 'DATE':
            entities['dates'].append(ent.text)
        elif ent.label_ == 'MONEY':
            entities['money'].append(ent.text)
    return entities

# Test
result = extract_entities(text)
for entity_type, values in result.items():
    print(f'{entity_type}: {values}')

persons: ['John Smith']
organizations: ['Acme Corporation']
locations: ['New York']
dates: ['10001', 'March 15, 2024']
money: ['1,250.50']


Visualize Entities with displaCy

In [10]:
# Visualize inside notebook (inline)
displacy.render(doc, style='ent', jupyter=True)

# Save as HTML file (for download/submission)
html_content = displacy.render(doc, style='ent', page=True, jupyter=False)
if html_content:
    with open('/kaggle/working/entities.html', 'w', encoding='utf-8') as f:
        f.write(html_content)
    print("✅ HTML saved to /kaggle/working/entities.html")
else:
    print("⚠️ displacy.render returned None. Check your spaCy version or doc object.")

NameError: name 'displacy' is not defined

Build Invoice Processor

In [ ]:
import json

def process_invoice_text(text):
    """Complete pipeline: Text → Extraction → JSON"""
    
    # Step 1: Extract with regex
    invoice_data = {
        'invoice_number': extract_invoice_number(text),
        'dates': extract_dates(text),
        'amounts': extract_amounts(text)
    }
    
    # Step 2: Extract with NER
    entities = extract_entities(text)
    invoice_data.update(entities)
    
    # Step 3: Post-process
    if invoice_data['amounts']:
        invoice_data['total_amount'] = max(invoice_data['amounts'])
    if invoice_data['dates']:
        invoice_data['invoice_date'] = invoice_data['dates'][0]
    
    return invoice_data

# Test on sample text
sample_text = """Invoice Number: INV-2024-001
Date: March 15, 2024
From: Acme Corporation
To: John Smith
Total Amount: $1,250.50"""

result = process_invoice_text(sample_text)
print(json.dumps(result, indent=2))

Save Results as JSON

In [ ]:
# Save to JSON file
output_file = 'extracted_data.json'
with open(output_file, 'w') as f:
    json.dump(result, f, indent=2)
print(f'Results saved to {output_file}')

# Read and verify
with open(output_file, 'r') as f:
    saved_data = json.load(f)
print("\nLoaded from JSON:")
print(json.dumps(saved_data, indent=2))

Install pytesseract (OCR)

In [ ]:
!apt-get install tesseract-ocr
!pip install pytesseract pillow

Complete OCR Pipeline 

In [ ]:
import pytesseract
from PIL import Image

def process_invoice_image(image_path):
    """Complete pipeline: OCR → Extraction → JSON"""
    
    # Step 1: OCR
    img = Image.open(image_path)
    text = pytesseract.image_to_string(img)
    print("Extracted Text:")
    print(text)
    print("-" * 50)
    
    # Step 2: Extract information
    return process_invoice_text(text)

# Test on image (agar hai to)
# result = process_invoice_image('sample_invoice.jpg')
# print(json.dumps(result, indent=2))